## Phase 2 – Model Training

In this phase, we focused on training different machine learning models using the preprocessed data from Phase 1.  
The main goal was to prepare and train our models so we can later evaluate them and decide which one works best for detecting phishing emails.

We decided to test three models that are commonly used and well-known for text classification:

1. **Logistic Regression** – a simple but powerful baseline model that usually performs very well with TF-IDF text data.  
2. **Linear SVM (Support Vector Machine)** – works great with high-dimensional features and can handle TF-IDF vectors efficiently.  
3. **Naive Bayes (optional)** – a lightweight model that trains quickly and provides a good point of comparison for more complex algorithms.

We chose these models because they are effective, easy to interpret, and well-suited for spam or phishing detection tasks.


### Data Loading and Splitting

Now we start by importing our data and splitting it into training and testing sets.    
We’re using the same data split for all models so that every model is trained and tested on identical data.      
This way, later when we compare their performance, the results will be fair and consistent.

In [9]:
# import the libraries we’ll need
import joblib
from sklearn.model_selection import train_test_split

# load the feature data from Phase 1 (TF-IDF + numeric features)
data = joblib.load("artifacts/feature_data.joblib")
X = data["X"]
y = data["y"]

print("Data loaded successfully")
print("X shape:", X.shape)
print("y shape:", y.shape)

# split our data into train/test once (we’ll use the same for all models)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " Test:", X_test.shape)

Data loaded successfully
X shape: (39139, 5015)
y shape: (39139,)
Train: (31311, 5015)  Test: (7828, 5015)


### Logistic Regression

We started with Logistic Regression because it’s often a strong baseline model for binary classification.  
It’s simple, fast, and works really well with large TF-IDF datasets.  
Here, we train it and show the results just to confirm that it’s working —  
We’ll explain these results in more detail later in the evaluation section of this phase.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV

# Create the Logistic Regression model
log_model = LogisticRegression(max_iter=2000, random_state=42)

# Define hyperparameter values to try
param_grid = {
    'C': [0.01, 0.1, 1, 10],       # Regularization strength
    'solver': ['liblinear', 'saga'] # Different solvers
}

# Set up GridSearchCV with 3-fold cross-validation
grid_log = GridSearchCV(log_model, param_grid, cv=3, scoring='accuracy', n_jobs=-1)

# Print training start message
print("Training Logistic Regression")  

# Train the model using Grid Search
grid_log.fit(X_train, y_train)
print("Model trained successfully")  # Keep the post-training message

# Print best parameters found during search
print("Best parameters:", grid_log.best_params_)
print("Best CV score:", round(grid_log.best_score_ * 100, 2), "%")

# Test the model on new/unseen data
y_pred = grid_log.predict(X_test)

# Compute accuracy and print classification report
acc = accuracy_score(y_test, y_pred)
print("Test Accuracy:", round(acc * 100, 2), "%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Training Logistic Regression
Model trained successfully
Best parameters: {'C': 10, 'solver': 'liblinear'}
Best CV score: 99.68 %
Test Accuracy: 99.76 %

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3462
           1       1.00      1.00      1.00      4366

    accuracy                           1.00      7828
   macro avg       1.00      1.00      1.00      7828
weighted avg       1.00      1.00      1.00      7828



### Linear SVM (Support Vector Machine)

Next, we trained a Linear SVM model.  
This model is great for high-dimensional data like our TF-IDF features.  
We also used Grid Search with Cross Validation to find the best hyperparameter “C” value and to make sure we don’t overfit.  
Again, we’re only checking that it trains and runs properly — we’ll analyze its performance later.

In [11]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# define our SVM model and parameter grid
param_grid = {'C': [0.1, 1, 10]}
svm = LinearSVC(max_iter=2000, random_state=42)

# use GridSearchCV to find the best C value
grid = GridSearchCV(svm, param_grid, cv=3, n_jobs=-1)
print("Training and tuning Linear SVM...")
grid.fit(X_train, y_train)

# print the best parameters and validation score
print("Best parameters:", grid.best_params_)
print("Best CV score:", round(grid.best_score_ * 100, 2), "%")

# test on unseen data
y_pred_svm = grid.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
print("Test Accuracy:", round(acc_svm * 100, 2), "%")

# quick report just to confirm it’s working
print("\n for verification only We’ll explain results in detail later in the evaluation section\n")
print(classification_report(y_test, y_pred_svm))



Training and tuning Linear SVM...
Best parameters: {'C': 10}
Best CV score: 99.8 %
Test Accuracy: 99.82 %

 for verification only We’ll explain results in detail later in the evaluation section

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3462
           1       1.00      1.00      1.00      4366

    accuracy                           1.00      7828
   macro avg       1.00      1.00      1.00      7828
weighted avg       1.00      1.00      1.00      7828



### Naive Bayes 

Lastly, we trained a Naive Bayes model as one more way to test our data.
It’s a simple and fast algorithm that uses probabilities to decide whether an email is phishing or not.
Even though it’s not as complex as the other models, it’s still helpful for seeing how a quick, lightweight method performs in comparison.

In [12]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

# Create Naive Bayes model
nb = MultinomialNB()

# Print training start message
print("Training Naive Bayes...")

# Define Grid Search with hyperparameter alpha
param_grid_nb = {'alpha': [0.1, 0.5, 1.0, 2.0]}

# Set up GridSearchCV with 3-fold cross-validation
grid_nb = GridSearchCV(nb, param_grid_nb, cv=3, scoring='accuracy', n_jobs=-1)

# Train the model using Grid Search
grid_nb.fit(X_train, y_train)
print("Model trained successfully!")

# Print best parameters found during search
print("Best parameters:", grid_nb.best_params_)
print("Best CV score:", round(grid_nb.best_score_ * 100, 2), "%")

# Test the model on unseen data
y_pred_nb = grid_nb.predict(X_test)

# Compute accuracy and print classification report
acc_nb = accuracy_score(y_test, y_pred_nb)
print("Test Accuracy:", round(acc_nb * 100, 2), "%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))


Training Naive Bayes...
Model trained successfully!
Best parameters: {'alpha': 2.0}
Best CV score: 99.42 %
Test Accuracy: 99.43 %

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3462
           1       1.00      0.99      0.99      4366

    accuracy                           0.99      7828
   macro avg       0.99      0.99      0.99      7828
weighted avg       0.99      0.99      0.99      7828

